# Bifrost.jl — visual demos

One notebook consolidating every visual demo that historically lived in
`test/human/demo*.jl`. Each section tells one story about the Bifrost API: the cells
exercising Bifrost are written to be read (duplication is deliberate), and all plotting
machinery lives in [`demo-helper.jl`](demo-helper.jl). The narrative for each demo —
its lesson, edge case, or invariant — is recorded in [`demo-intent.md`](demo-intent.md).

**Running**: use any Julia 1.11 kernel; the setup cell activates the
`test/human` environment (`Project.toml` next to this notebook) and brings in the
helper. Plots are interactive Plotly figures rendered inline.

| § | Class | Legacy source |
| --- | --- | --- |
| 1 | Smallest runnable example | `demo-smallest.jl` |
| 2 | Path geometry | `demo-path-geometry.jl` |
| 3 | Modify — field-level MCM meta | `demo1.jl` |
| 4 | Material spin | `demo1.jl` |
| 5 | Adaptive step-doubling | `demo1.jl` |
| 6 | JumpBy / JumpTo — 2D sweeps | `demo2.jl` |
| 7 | Meta × JumpTo interplay | `demo2.jl` |
| 8 | JumpBy / JumpTo — 3D scenes | `demo2.jl` |
| 9 | MCM temperature PTF | `demo3mcm.jl` |
| 10 | MCM speed benchmarks | `demo3benchmark.jl` |

In [ ]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

using Bifrost
using MonteCarloMeasurements
include("demo-helper.jl")
using .DemoHelper

## 1 · The smallest runnable example

The full layer stack — material → cross-section → geometry → fiber → propagation —
in a dozen lines. The 90° bend at R = 0.05 m introduces bend birefringence with a
fixed axis, so `J` comes out diagonal, `diag(e^{-iφ}, e^{+iφ})` with φ ≈ 0.033 rad: a
pure lossless retarder (SU(2)). `stats` shows the propagation decomposed into 3
intervals — the propagator never steps across the straight/bend segment boundaries
(the breakpoint invariant).

In [ ]:
xs = StepIndexCrossSection(
    SilicaGermaniaGlass(0.036),
    SilicaGermaniaGlass(0.0),
    8.2e-6,
    125e-6;
    manufacturer = "Corning",
    model_number = "SMF-like",
)

# Single Subpath: lead-in straight + 90° bend (axis_angle = 0) + lead-out straight.
# seal! ends the Subpath at its natural exit with no terminal connector bending.
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5, meta = [Nickname("lead-in")])
bend!(sb;     radius = 0.05, angle = π / 2, meta = [Nickname("90° bend")])
straight!(sb; length = 0.5, meta = [Nickname("lead-out")])
seal!(sb)

fiber = Fiber(build(sb); cross_section = xs, T_ref_K = 297.15)
J, stats = propagate_fiber(fiber; λ_m = 1550e-9)

println("J =")
display(J)
println("intervals = ", length(stats))

## 2 · Path geometry

The geometry layer (`geometry/path-geometry.jl`) builds and queries three-dimensional
paths with no optics attached. Authoring follows one lifecycle on a `SubpathBuilder`:
`start!` → segment calls (`straight!`, `bend!`, `helix!`, `catenary!`, `jumpby!`) →
seal (`jumpto!` to a global target, or `seal!` at the natural exit) → `build`.

Every demo in this section renders through the **path inspector**
(`path_inspector` in `demo-helper.jl`): the centerline with arc-length-graded
markers, open circles at segment joins, green start / red end dots, and `Nickname`
labels. The **slider scrubs arc length** along the path, carrying a translucent
normal–binormal plane, the local T̂/N̂/B̂ frame triad (orange/blue/green), and a red
in-plane arrow at the accumulated spin phase; the readout below the scene reports
s; x, y, z; curvature κ; geometric torsion τ_geom; spin rate τ_spin; and ∫τ_spin ds.

### 2.1 · A simple multi-segment Subpath

All four curve primitives composed in one Subpath — straight · 90° bend · straight ·
catenary sag · 60° bend · straight — sealed at its natural exit. The lesson: segments
join with tangent (G1) continuity automatically, and the built path answers scalar
geometry queries (`path_length`, `writhe` — zero here, the path is planar).

In [ ]:
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.10, meta = [Nickname("Straight")])
bend!(sb;     radius = 0.05, angle = π / 2, meta = [Nickname("Bend")])
straight!(sb; length = 0.12, meta = [Nickname("Straight")])
catenary!(sb; a = 0.03, length = 0.10, axis_angle = 0.0, meta = [Nickname("Catenary")])
bend!(sb;     radius = 0.06, angle = π / 3, meta = [Nickname("Bend")])
straight!(sb; length = 0.08, meta = [Nickname("Straight")])
seal!(sb)
path = build(sb)

println("Arc length: ", path_length(path), " m")
println("Writhe:     ", writhe(path; n = 128))
path_inspector(path; title = "Single Subpath: straights, bends, catenary")

### 2.2 · Segment nicknames

The same spirit with a helix added, and every segment authored with a *descriptive*
`Nickname` (`lead-in`, `sag`, `spin section`, …). The lesson: names live with the
segment definition and flow through the build to presentation — nothing derives
labels from type information at render time.

In [ ]:
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.08, meta = [Nickname("lead-in")])
bend!(sb;     radius = 0.06, angle = π / 2, meta = [Nickname("90° bend")])
straight!(sb; length = 0.06, meta = [Nickname("spacer")])
catenary!(sb; a = 0.04, length = 0.08, axis_angle = 0.0, meta = [Nickname("sag")])
helix!(sb;    radius = 0.025, pitch = 0.015, turns = 1.2, axis_angle = 0.0,
              meta = [Nickname("spin section")])
straight!(sb; length = 0.06, meta = [Nickname("lead-out")])
seal!(sb)
path = build(sb)

println("Arc length: ", path_length(path), " m")
path_inspector(path; title = "Path geometry: segment nicknames")

### 2.3 · Helix `axis_angle`

A 2-turn helix between two straights, with `axis_angle` ∈ {0, π/3, 2π/3}.
`axis_angle` rotates the helix axis about the incoming tangent: entry stays
tangent-continuous for every value, while the exit direction — and where the
lead-out straight goes — swings around. The arc length is invariant under the
rotation, as the printout confirms.

*(The legacy demos rendered these as three separate files; here the comparison row
makes the lesson visible at a glance, and the inspector below it lets you scrub the
most out-of-plane variant in detail.)*

In [ ]:
function helix_path(axis_angle)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 0.05, meta = [Nickname("lead-in")])
    helix!(sb;    radius = 0.03, pitch = 0.02, turns = 2.0, axis_angle = axis_angle,
                  meta = [Nickname("helix")])
    straight!(sb; length = 0.05, meta = [Nickname("lead-out")])
    seal!(sb)
    return build(sb)
end

for aa in (0.0, π / 3, 2π / 3)
    println("axis_angle = ", round(aa; digits = 4),
            ":  arc length = ", round(path_length(helix_path(aa)); digits = 4), " m")
end

variant_row([("axis_angle = 0",    helix_path(0.0),    [2]),
             ("axis_angle = π/3",  helix_path(π / 3),  [2]),
             ("axis_angle = 2π/3", helix_path(2π / 3), [2])];
            spacing = 0.25,
            title = "HelixSegment axis_angle: same entry tangency, rotated axis")

In [ ]:
path_inspector(helix_path(2π / 3); title = "HelixSegment: axis_angle = 2π/3")

### 2.4 · The paddle: five Subpaths, `:inherit`, and `min_bend_radius`

A `PathBuilt` of five Subpaths forming a paddle pattern: vertical 1 m straights at
x = 0, 1, 2, 3 alternating up/down, joined by terminal `jumpto!` connectors with
anti-parallel landing tangents (180° U-turns), each with its own `min_bend_radius`.
Subpath 4 also carries an *interior* `jumpby!` — its `delta` is expressed in the
local frame (after heading −z, local (−1, 0, 0) lands at the global (2, 0, 0)) —
before its sealing `jumpto!`. Subpaths 2–5 start with `start!(sb, :inherit)`, so
each start state flows from the previous subpath's endpoint instead of being
hand-loaded.

In [ ]:
# Subpath 1: straight up to (0,0,1); U-turn connector lands at (1,0,1) heading -z.
sb1 = SubpathBuilder(); start!(sb1)
straight!(sb1; length = 1.0, meta = [Nickname("Sub1 straight")])
jumpto!(sb1; point = (1.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.4)

# Subpath 2: inherits (1,0,1) heading -z; straight down; U-turn to (2,0,0) heading +z.
sb2 = SubpathBuilder(); start!(sb2, :inherit)
straight!(sb2; length = 1.0, meta = [Nickname("Sub2 straight")])
jumpto!(sb2; point = (2.0, 0.0, 0.0), incoming_tangent = (0.0, 0.0, 1.0),
        min_bend_radius = 0.1)

# Subpath 3: inherits (2,0,0) heading +z; straight up; U-turn to (3,0,1) heading -z.
sb3 = SubpathBuilder(); start!(sb3, :inherit)
straight!(sb3; length = 1.0, meta = [Nickname("Sub3 straight")])
jumpto!(sb3; point = (3.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.05)

# Subpath 4: inherits (3,0,1) heading -z; straight down, then an INTERIOR JumpBy
# whose local-frame delta (-1,0,0) lands at the global (2,0,0) heading +z; the
# sealing jumpto! pins that landing point exactly.
sb4 = SubpathBuilder(); start!(sb4, :inherit)
straight!(sb4; length = 1.0, meta = [Nickname("Sub4 straight")])
jumpby!(sb4; delta = (-1.0, 0.0, 0.0), tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.1, meta = [Nickname("Sub4 JumpBy")])
jumpto!(sb4; point = (2.0, 0.0, 0.0), incoming_tangent = (0.0, 0.0, 1.0))

# Subpath 5: inherits (2,0,0) heading +z and runs straight out.
sb5 = SubpathBuilder(); start!(sb5, :inherit)
straight!(sb5; length = 1.0, meta = [Nickname("Sub5 straight")])
seal!(sb5)

paddle = build([sb1, sb2, sb3, sb4, sb5])
println("PathBuilt arc length: ", path_length(paddle), " m")
path_inspector(paddle; fidelity = 4.0,
               title = "JumpBy/JumpTo paddle: PathBuilt of 5 Subpaths")

### 2.5 · PathBuilt assembly with exact handoffs

Three Subpaths — a straight sealed by `jumpto!` at (0, 0, 0.2), an `:inherit`
quarter-bend sealed by `jumpto!` at its analytic exit (R, 0, 0.2 + R), and an
`:inherit` helix sealed with `seal!` at its natural exit. A `jumpto!` seal pins
exact handoff coordinates, and the build's conformity check validates every
boundary. Per-Subpath `Nickname` meta labels whole subpaths.

In [ ]:
sb1 = SubpathBuilder(meta = [Nickname("Subpath 1: straight")])
start!(sb1)
straight!(sb1; length = 0.2, meta = [Nickname("Straight")])
jumpto!(sb1; point = (0.0, 0.0, 0.2), incoming_tangent = (0.0, 0.0, 1.0))

# Quarter bend from +z to +x: exits at (R, 0, 0.2 + R) heading +x.
R = 0.05
sb2 = SubpathBuilder(meta = [Nickname("Subpath 2: bend")])
start!(sb2, :inherit)
bend!(sb2; radius = R, angle = π / 2, meta = [Nickname("90° bend")])
jumpto!(sb2; point = (R, 0.0, 0.2 + R), incoming_tangent = (1.0, 0.0, 0.0))

sb3 = SubpathBuilder(meta = [Nickname("Subpath 3: helix")])
start!(sb3, :inherit)
helix!(sb3; radius = 0.025, pitch = 0.02, turns = 1.5, axis_angle = 0.0,
       meta = [Nickname("Helix")])
seal!(sb3)

path = build([sb1, sb2, sb3])
println("Subpaths:   ", length(path.subpaths))
println("Arc length: ", path_length(path), " m")
path_inspector(path; fidelity = 2.0,
               title = "PathBuilt: three Subpaths (straight, bend, helix)")

## 3 · Modify — field-level MCM meta on segment fields

Twelve structurally identical experiments showing what each geometry field of a
segment *means*, by perturbing exactly one of them. The baseline is a 3-segment
inverted-U (straight 1 m · π-bend R = 0.5 · straight 1 m), or a 4-segment variant
with a helix (R = 0.15, pitch = 0.25, turns = 1.5) inserted after the bend for the
helix targets. One segment carries one `MCMadd(:field, δ)` or `MCMmul(:field, f)`
with a plain `Float64` value — the MCM machinery applied with deterministic offsets,
so the geometric effect is visible to the eye. `Fiber(builder; ...)` applies the
meta during its single build.

In every row the perturbed segment draws **red**, fixed segments **green**, and the
terminal connector faint gray. Watch how everything downstream of the perturbed
segment rides along rigidly — there is no anchor; contrast §7 where a `jumpto!`
pins the endpoint.

In [ ]:
DEMO_XS = StepIndexCrossSection(
    SilicaGermaniaGlass(0.036),
    SilicaGermaniaGlass(0.0),
    8.2e-6,
    125e-6,
)
DEMO_T_K = 297.15

# Inverted-U baseline; `target_meta` attaches to segment `target_idx`
# (1 = first straight, 2 = π-bend, 3 = second straight).
function modify_variant(target_idx, target_meta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0, meta = target_idx == 1 ? target_meta : AbstractMeta[])
    bend!(sb; radius = 0.5, angle = π, axis_angle = 0.0,
          meta = target_idx == 2 ? target_meta : AbstractMeta[])
    straight!(sb; length = 1.0, meta = target_idx == 3 ? target_meta : AbstractMeta[])
    seal!(sb)
    return Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path
end

# 4-segment baseline with a helix between bend and second straight
# (1 = straight, 2 = bend, 3 = helix, 4 = straight).
function modify_variant_helix(target_idx, target_meta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0, meta = target_idx == 1 ? target_meta : AbstractMeta[])
    bend!(sb; radius = 0.5, angle = π, axis_angle = 0.0,
          meta = target_idx == 2 ? target_meta : AbstractMeta[])
    helix!(sb; radius = 0.15, pitch = 0.25, turns = 1.5, axis_angle = 0.0,
           meta = target_idx == 3 ? target_meta : AbstractMeta[])
    straight!(sb; length = 1.0, meta = target_idx == 4 ? target_meta : AbstractMeta[])
    seal!(sb)
    return Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path
end
println("builders defined")

### 3.1 · `MCMadd` on the inverted-U

Additive offsets: shortening the first straight translates everything downstream;
offsetting the bend radius widens/narrows the U; offsetting the bend angle changes
the exit direction — `+π` closes a full circle.

In [ ]:
variant_row([("baseline",              modify_variant(1, AbstractMeta[]), Int[]),
             ("MCMadd(:length, -0.4)", modify_variant(1, [MCMadd(:length, -0.4)]), [1])];
            spacing = 2.5, gray_terminal = true,
            title = "MCMadd(:length) on the first straight")

In [ ]:
variant_row([("baseline",               modify_variant(2, AbstractMeta[]), Int[]),
             ("MCMadd(:radius, -0.25)", modify_variant(2, [MCMadd(:radius, -0.25)]), [2]),
             ("MCMadd(:radius, +0.50)", modify_variant(2, [MCMadd(:radius,  0.50)]), [2])];
            spacing = 2.5, gray_terminal = true,
            title = "MCMadd(:radius) on the bend")

In [ ]:
variant_row([("baseline",             modify_variant(2, AbstractMeta[]), Int[]),
             ("MCMadd(:angle, -π/2)", modify_variant(2, [MCMadd(:angle, -π / 2)]), [2]),
             ("MCMadd(:angle, +π/4)", modify_variant(2, [MCMadd(:angle,  π / 4)]), [2]),
             ("MCMadd(:angle, +π)",   modify_variant(2, [MCMadd(:angle,  π)]),     [2])];
            spacing = 2.5, gray_terminal = true,
            title = "MCMadd(:angle) on the bend — +π closes the circle")

### 3.2 · `MCMmul` on the inverted-U

Multiplicative scaling — and the sign-flip edge case: `MCMmul(:length, -0.4)`
makes the first straight walk *backward* along the local tangent.

In [ ]:
variant_row([("baseline",              modify_variant(1, AbstractMeta[]), Int[]),
             ("MCMmul(:length, -0.4)", modify_variant(1, [MCMmul(:length, -0.4)]), [1]),
             ("MCMmul(:length, +0.5)", modify_variant(1, [MCMmul(:length,  0.5)]), [1])];
            spacing = 2.5, gray_terminal = true,
            title = "MCMmul(:length) on the first straight — negative walks backward")

In [ ]:
variant_row([("baseline",             modify_variant(2, AbstractMeta[]), Int[]),
             ("MCMmul(:radius, 0.5)", modify_variant(2, [MCMmul(:radius, 0.5)]), [2]),
             ("MCMmul(:radius, 2.0)", modify_variant(2, [MCMmul(:radius, 2.0)]), [2])];
            spacing = 2.5, gray_terminal = true,
            title = "MCMmul(:radius) on the bend")

In [ ]:
variant_row([("baseline",             modify_variant(2, AbstractMeta[]), Int[]),
             ("MCMmul(:angle, 0.5)",  modify_variant(2, [MCMmul(:angle, 0.5)]),  [2]),
             ("MCMmul(:angle, 1.25)", modify_variant(2, [MCMmul(:angle, 1.25)]), [2])];
            spacing = 2.5, gray_terminal = true,
            title = "MCMmul(:angle) on the bend")

### 3.3 · Helix fields

The three helix knobs have visibly different geometry: `:radius` fattens the coil
(pitch fixed), `:pitch` stretches it axially, and `:turns` changes the winding count
and therefore the exit phase and direction.

In [ ]:
variant_row([("baseline",               modify_variant_helix(3, AbstractMeta[]), Int[]),
             ("MCMadd(:radius, -0.05)", modify_variant_helix(3, [MCMadd(:radius, -0.05)]), [3]),
             ("MCMadd(:radius, +0.10)", modify_variant_helix(3, [MCMadd(:radius,  0.10)]), [3])];
            spacing = 3.5, gray_terminal = true,
            title = "MCMadd(:radius) on the helix")

In [ ]:
variant_row([("baseline",              modify_variant_helix(3, AbstractMeta[]), Int[]),
             ("MCMadd(:pitch, -0.10)", modify_variant_helix(3, [MCMadd(:pitch, -0.10)]), [3]),
             ("MCMadd(:pitch, +0.20)", modify_variant_helix(3, [MCMadd(:pitch,  0.20)]), [3])];
            spacing = 3.5, gray_terminal = true,
            title = "MCMadd(:pitch) on the helix")

In [ ]:
variant_row([("baseline",             modify_variant_helix(3, AbstractMeta[]), Int[]),
             ("MCMadd(:turns, -0.5)", modify_variant_helix(3, [MCMadd(:turns, -0.5)]), [3]),
             ("MCMadd(:turns, +0.5)", modify_variant_helix(3, [MCMadd(:turns,  0.5)]), [3])];
            spacing = 3.5, gray_terminal = true,
            title = "MCMadd(:turns) on the helix")

In [ ]:
variant_row([("baseline",             modify_variant_helix(3, AbstractMeta[]), Int[]),
             ("MCMmul(:radius, 0.5)", modify_variant_helix(3, [MCMmul(:radius, 0.5)]), [3]),
             ("MCMmul(:radius, 2.0)", modify_variant_helix(3, [MCMmul(:radius, 2.0)]), [3])];
            spacing = 3.5, gray_terminal = true,
            title = "MCMmul(:radius) on the helix")

In [ ]:
variant_row([("baseline",            modify_variant_helix(3, AbstractMeta[]), Int[]),
             ("MCMmul(:pitch, 0.5)", modify_variant_helix(3, [MCMmul(:pitch, 0.5)]), [3]),
             ("MCMmul(:pitch, 2.0)", modify_variant_helix(3, [MCMmul(:pitch, 2.0)]), [3])];
            spacing = 3.5, gray_terminal = true,
            title = "MCMmul(:pitch) on the helix")

In [ ]:
variant_row([("baseline",             modify_variant_helix(3, AbstractMeta[]), Int[]),
             ("MCMmul(:turns, 0.67)", modify_variant_helix(3, [MCMmul(:turns, 0.67)]), [3]),
             ("MCMmul(:turns, 1.5)",  modify_variant_helix(3, [MCMmul(:turns, 1.5)]),  [3])];
            spacing = 3.5, gray_terminal = true,
            title = "MCMmul(:turns) on the helix")

## 4 · Material spin

`start!(sb; spin_rate = 2π)` sets a constant material spin of one full turn per
meter over the whole Subpath. Scrub the slider: the readout shows
τ_spin = 6.2832 rad/m everywhere and ∫τ_spin ds accumulating linearly with s, while
the red spin arrow rotates in the normal–binormal plane — independently of the
helix's geometric torsion, which the T̂/N̂/B̂ triad follows. The lesson: material
spin is resolved by the geometry layer into path-coordinate spin, carried separately
from the frame geometry.

In [ ]:
sb = SubpathBuilder(); start!(sb; spin_rate = 2π)
straight!(sb; length = 1.0, meta = [Nickname("lead-in")])
helix!(sb; radius = 0.5, pitch = 0.05, turns = 4.0, axis_angle = 0.0,
       meta = [Nickname("helix")])
straight!(sb; length = 1.0, meta = [Nickname("lead-out")])
seal!(sb)
path = build(sb)

path_inspector(path; title = "Helix with material spin (2π rad/m)")

## 5 · Adaptive step-doubling diagnostic

A solver diagnostic on a smooth, *noncommuting* generator (not a fiber):

$$K(s) = α\,i\,σ_x\cos(πs) + β\,i\,σ_z\sin(2πs), \qquad α = 1.2,\; β = 0.9,\; s ∈ [0, 2]$$

`collect_adaptive_steps` (a recording mirror of the production controller in
`path-integral.jl` — the solver itself is untouched) returns every accepted and
rejected trial step. What to look for: the step size locks onto the local error
budget and grows where ‖K(s)‖ dips; each over-grown step triggers a short rejection
cascade (red ✕) right after the ‖K‖ minima; in the middle panel the err/tol ratio
hugs the threshold from below — the controller wastes little budget — and no
accepted step exceeds it.

In [ ]:
using Bifrost.Plots: collect_adaptive_steps
using LinearAlgebra

SX = ComplexF64[0 1; 1 0]
SZ = ComplexF64[1 0; 0 -1]
α, β = 1.2, 0.9
K = s -> α * im * cos(π * s) .* SX + β * im * sin(2π * s) .* SZ

J0 = Matrix{ComplexF64}(I, 2, 2)
J, records = collect_adaptive_steps(K, 0.0, 2.0, J0; rtol = 1e-6, atol = 1e-9)

println("accepted: ", count(r -> r.accepted, records),
        "   rejected: ", count(r -> !r.accepted, records))

adaptive_panels(records, s -> opnorm(K(s)), 0.0, 2.0;
    rtol = 1e-6, atol = 1e-9,
    components = [("α·cos(πs)  [σx coeff]", s -> α * cos(π * s),  "#1f77b4"),
                  ("β·sin(2πs) [σz coeff]", s -> β * sin(2π * s), "#ff7f0e")],
    title = "Adaptive step-doubling on a noncommuting K(s)")

## 6 · JumpBy / JumpTo — 2D sweeps

Connectors bridge gaps with G2 quintic curves. The defining difference:

- **`jumpby!`** is an *interior* segment whose `delta` (and optional outgoing
  `tangent`) live in the **local frame** at the jump's start;
- **`jumpto!`** *seals* a Subpath with a connector to a **global**-frame `point`
  (with optional global `incoming_tangent` / `incoming_curvature`); the next
  Subpath continues from there via `start!(sb, :inherit)`.

Each sweep below varies exactly one connector degree of freedom in an otherwise
identical scene — straight (1 m) · connector (red) · straight (1 m) — with all paths
in the y = 0 plane, viewed in the x–z projection.

### 6.1 · JumpBy: outgoing tangent (local frame)

Same `delta = (0.4, 0, 0.4)`; the outgoing tangent swings the exit straight from
+45° through axial to −45°.

In [ ]:
function jumpby_tangent_path(tangent)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    jumpby!(sb; delta = (0.4, 0.0, 0.4), tangent = tangent)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

variant_row_2d(
    [("t = (+1,0,1)/√2", jumpby_tangent_path((1 / sqrt(2), 0.0, 1 / sqrt(2))), [2]),
     ("t = (0,0,1)",     jumpby_tangent_path((0.0, 0.0, 1.0)),                 [2]),
     ("t = (-1,0,1)/√2", jumpby_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.0, title = "JumpBy: outgoing tangent sweep (local frame)")

### 6.2 · JumpTo: target point (global frame)

The target is a lab-frame waypoint at z = 1.5; sweeping its transverse offset bends
the connector progressively harder.

In [ ]:
function jumpto_point_path(point)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = point)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

variant_row_2d(
    [("x = $(x)", jumpto_point_path((x, 0.0, 1.5)), [2]) for x in (0.0, 0.3, 0.6, 1.0)];
    spacing = 2.5, title = "JumpTo(point = (x, 0, 1.5)): transverse sweep")

### 6.3 · JumpTo: incoming tangent (global frame)

Fixed landing point, three different global landing directions — the connector
arrives at the same point from three headings and the next straight leaves along
each.

In [ ]:
function jumpto_tangent_path(incoming_tangent)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = (0.5, 0.0, 1.5), incoming_tangent = incoming_tangent)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

variant_row_2d(
    [("incoming = (+1,0,0)",    jumpto_tangent_path((1.0, 0.0, 0.0)),                  [2]),
     ("incoming = (0,0,+1)",    jumpto_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("incoming = (-1,0,1)/√2", jumpto_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.5, title = "JumpTo: incoming tangent sweep (global frame)")

### 6.4 · JumpTo routing: a serpentine of waypoints

Three successive `jumpto!`s with anti-parallel landing tangents route the path
through (1,0,1) → (2,0,0) → (3,0,1). Each `jumpto!` seals a Subpath, so the chain is
a 3-Subpath `PathBuilt` under the hood — connectors red, straights green.

In [ ]:
sb1 = SubpathBuilder(); start!(sb1)
straight!(sb1; length = 1.0)
jumpto!(sb1; point = (1.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.1)
sb2 = SubpathBuilder(); start!(sb2, :inherit)
straight!(sb2; length = 1.0)
jumpto!(sb2; point = (2.0, 0.0, 0.0), incoming_tangent = (0.0, 0.0, 1.0),
        min_bend_radius = 0.1)
sb3 = SubpathBuilder(); start!(sb3, :inherit)
straight!(sb3; length = 1.0)
jumpto!(sb3; point = (3.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
        min_bend_radius = 0.1)
routing = build([sb1, sb2, sb3])

variant_row_2d([("composite", routing, [2, 4, 6])];
               spacing = 0.0, title = "JumpTo routing: serpentine through waypoints")

### 6.5 · JumpTo `min_bend_radius`: a feasibility sweep

The canonical infeasibility case: a transverse unit chord with anti-parallel landing
tangent (a 180° U-turn). `min_bend_radius` is a hard build-time guardrail — variants
that violate it throw at `build` and render here as the surviving lead-in only,
labeled *(infeasible)*.

The legacy sweep values bracket 0.5 m = chord/2, the **circular-arc ideal** for this
U-turn. Observed behavior: 0.49 is *also* infeasible — the quintic connector's peak
curvature necessarily exceeds the circular bound, so the true threshold sits between
0.30 and 0.49.

In [ ]:
function mbr_path(mbr)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = (1.0, 0.0, 1.0), incoming_tangent = (0.0, 0.0, -1.0),
            min_bend_radius = mbr)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

lead_in_only() = begin
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    seal!(sb)
    build(sb)
end

variants = map((0.10, 0.30, 0.49, 0.51, 0.70)) do mbr
    path, err = try_build(() -> mbr_path(mbr))
    err === nothing ? ("mbr = $(mbr)", path, [2]) :
                      ("mbr = $(mbr) (infeasible)", lead_in_only(), Int[])
end

variant_row_2d(collect(variants); spacing = 2.0,
    title = "JumpTo min_bend_radius sweep — U-turn over a transverse unit chord")

## 7 · Meta × JumpTo interplay — anchored vs unanchored perturbations

When upstream geometry is perturbed, what happens downstream? Three overlays
(baseline black, modified red; lengths and Δ% in the legend) answer it:

1. with a `jumpby!` there is **no anchor** — the endpoint drifts;
2. a `jumpto!` **pins** the endpoint while its connector absorbs the slack;
3. `:T_K` on the `jumpto!` seal itself thermally expands the connector *without*
   moving the anchor (the length-constrained connector resolve).

### 7.1 · S-curve + JumpBy: downstream drifts

An S-curve (two opposed 90° bends) followed by a fiber-relative `jumpby!`. Inflating
both bend radii ×1.25 grows the path ~10% — and since nothing anchors the tail, the
entire downstream trajectory shifts and the endpoint visibly separates.

In [ ]:
sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π)
straight!(sb; length = 0.3)
jumpby!(sb; delta = (0.0, 0.0, 0.8))
straight!(sb; length = 1.0)
seal!(sb)
baseline = build(sb)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0, meta = [MCMmul(:radius, 1.25)])
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π,   meta = [MCMmul(:radius, 1.25)])
straight!(sb; length = 0.3)
jumpby!(sb; delta = (0.0, 0.0, 0.8))
straight!(sb; length = 1.0)
seal!(sb)
modified = Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path

overlay_compare(baseline, modified; legend = :bottomright,
    title = "S-curve + JumpBy: no anchor — downstream drifts")

### 7.2 · S-curve + JumpTo: the anchor pins the endpoint

The same S-curve sealed by a `jumpto!` to a fixed lab-frame waypoint (placed so the
baseline totals 4.000 m). A stronger ×1.5 radius perturbation swings the interior
wide — the connector chord changes — but the endpoint stays pinned.

In [ ]:
# Probe build to place the anchor: same interior, sealed at its natural exit.
probe = SubpathBuilder(); start!(probe)
straight!(probe; length = 0.3)
bend!(probe; radius = 0.5, angle = π / 2, axis_angle = 0.0)
bend!(probe; radius = 0.5, angle = π / 2, axis_angle = π)
straight!(probe; length = 0.3)
seal!(probe)
pre = build(probe)
p_end = end_point(pre)
anchor = (p_end[1], p_end[2], p_end[3] + (4.0 - path_length(pre)))

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π)
straight!(sb; length = 0.3)
jumpto!(sb; point = anchor)
baseline = build(sb)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.3)
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0, meta = [MCMmul(:radius, 1.5)])
bend!(sb; radius = 0.5, angle = π / 2, axis_angle = π,   meta = [MCMmul(:radius, 1.5)])
straight!(sb; length = 0.3)
jumpto!(sb; point = anchor)
modified = Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path

overlay_compare(baseline, modified;
    title = "S-curve + JumpTo: anchor pins the endpoint, connector absorbs slack")

### 7.3 · `:T_K` on the JumpTo seal: the connector itself expands

A lead-in straight and its sealing `jumpto!` both carry `MCMadd(:T_K, ΔT)`, with ΔT
chosen for a 5% thermal expansion (via the cladding CTE at the reference
temperature). The terminal connector's arc length scales by τ while still landing on
the fixed point — the extra length shows up as connector curvature. Interpretation
of `:T_K` happens in `Fiber`; the geometry layer only carries it.

In [ ]:
α_lin = cte(DEMO_XS.cladding_material, DEMO_T_K)
ΔT  = 0.05 / α_lin            # 5% length growth
mdt = [MCMadd(:T_K, ΔT)]

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5)
jumpto!(sb; point = (1.0, 0.0, 0.5), incoming_tangent = (1.0, 0.0, 0.0))
baseline = build(sb)

sb = SubpathBuilder(); start!(sb)
straight!(sb; length = 0.5, meta = mdt)
jumpto!(sb; point = (1.0, 0.0, 0.5), incoming_tangent = (1.0, 0.0, 0.0), meta = mdt)
modified = Fiber(sb; cross_section = DEMO_XS, T_ref_K = DEMO_T_K).path

overlay_compare(baseline, modified;
    title = ":T_K on segment + JumpTo seal: connector thermally expands, anchor holds")

## 8 · JumpBy / JumpTo — 3D scenes

The §6 lessons restaged in 3D, plus the G2-continuity content that needs the third
dimension: prescribed connector curvature at either end, frame rotation after a
bend, and curvature inheritance. Connector red, fixed segments green, variants
offset along +x.

### 8.1 · JumpBy delta sweeps (local frame)

Axial `delta = (0, 0, d)` bridges a straight-through gap; transverse
`delta = (d, 0, 0.5)` doglegs sideways.

In [ ]:
function jumpby_delta_path(delta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    jumpby!(sb; delta = delta)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

variant_row([("d = $(d)", jumpby_delta_path((0.0, 0.0, d)), [2]) for d in (0.3, 0.6, 1.0)];
            spacing = 2.5, title = "JumpBy(delta = (0, 0, d)) — axial sweep")

In [ ]:
variant_row([("d = $(d)", jumpby_delta_path((d, 0.0, 0.5)), [2]) for d in (0.0, 0.2, 0.5)];
            spacing = 2.5, title = "JumpBy(delta = (d, 0, 0.5)) — transverse sweep")

### 8.2 · JumpBy: outgoing tangent and outgoing curvature (G2 exit knob)

With `curvature_out` the connector *leaves* with prescribed curvature — the (0,±2,0)
variants bow out of the x–z plane.

In [ ]:
variant_row(
    [("t = (+1,0,1)/√2", jumpby_tangent_path((1 / sqrt(2), 0.0, 1 / sqrt(2))),  [2]),
     ("t = (0,0,1)",     jumpby_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("t = (-1,0,1)/√2", jumpby_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.5, title = "JumpBy — outgoing tangent (local frame)")

In [ ]:
function jumpby_curvature_path(curvature_out)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    jumpby!(sb; delta = (0.5, 0.0, 0.5), tangent = (1.0, 0.0, 1.0) ./ sqrt(2),
            curvature_out = curvature_out)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

variant_row(
    [("κ_out = 0",        jumpby_curvature_path((0.0, 0.0, 0.0)),  [2]),
     ("κ_out = (0,+2,0)", jumpby_curvature_path((0.0, 2.0, 0.0)),  [2]),
     ("κ_out = (0,-2,0)", jumpby_curvature_path((0.0, -2.0, 0.0)), [2])];
    spacing = 2.5, title = "JumpBy — outgoing curvature (G2 exit knob)")

### 8.3 · JumpTo point and landing sweeps (global frame)

In [ ]:
variant_row([("z = $(z)", jumpto_point_path((0.0, 0.0, z)), [2]) for z in (1.3, 1.6, 2.0)];
            spacing = 2.5, title = "JumpTo(point = (0, 0, z)) — axial sweep")

In [ ]:
variant_row([("x = $(x)", jumpto_point_path((x, 0.0, 1.5)), [2]) for x in (0.0, 0.3, 0.6)];
            spacing = 2.5, title = "JumpTo(point = (x, 0, 1.5)) — transverse sweep")

In [ ]:
variant_row(
    [("incoming = (+1,0,0)",    jumpto_tangent_path((1.0, 0.0, 0.0)),                  [2]),
     ("incoming = (0,0,+1)",    jumpto_tangent_path((0.0, 0.0, 1.0)),                  [2]),
     ("incoming = (-1,0,1)/√2", jumpto_tangent_path((-1 / sqrt(2), 0.0, 1 / sqrt(2))), [2])];
    spacing = 2.5, title = "JumpTo — incoming tangent (global frame)")

In [ ]:
function jumpto_curvature_path(incoming_curvature)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    jumpto!(sb1; point = (0.5, 0.0, 1.5), incoming_tangent = (0.0, 0.0, 1.0),
            incoming_curvature = incoming_curvature)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

variant_row(
    [("κ_in = 0",         jumpto_curvature_path((0.0, 0.0, 0.0)),   [2]),
     ("κ_in = (+10,0,0)", jumpto_curvature_path((10.0, 0.0, 0.0)),  [2]),
     ("κ_in = (-10,0,0)", jumpto_curvature_path((-10.0, 0.0, 0.0)), [2])];
    spacing = 1.5, title = "JumpTo — incoming curvature (G2 landing knob)")

### 8.4 · After a bend: rotated local frame vs global target

After a 90° bend the local frame has rotated. A `jumpby!` delta is expressed in that
**rotated** frame — the same delta now points along the new heading. A `jumpto!`
target stays **global** — it does not rotate with the path.

In [ ]:
function jumpby_after_bend_path(delta)
    sb = SubpathBuilder(); start!(sb)
    straight!(sb; length = 1.0)
    bend!(sb; radius = 0.5, angle = π / 2, axis_angle = 0.0)
    jumpby!(sb; delta = delta)
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

variant_row(
    [("delta = (0,0,0.5)",    jumpby_after_bend_path((0.0, 0.0, 0.5)),  [3]),
     ("delta = (0.3,0,0.5)",  jumpby_after_bend_path((0.3, 0.0, 0.5)),  [3]),
     ("delta = (-0.3,0,0.5)", jumpby_after_bend_path((-0.3, 0.0, 0.5)), [3])];
    spacing = 3.5, title = "JumpBy after 90° bend — delta in the ROTATED local frame")

In [ ]:
function jumpto_after_bend_path(point)
    sb1 = SubpathBuilder(); start!(sb1)
    straight!(sb1; length = 1.0)
    bend!(sb1; radius = 0.5, angle = π / 2, axis_angle = 0.0)
    jumpto!(sb1; point = point)
    sb2 = SubpathBuilder(); start!(sb2, :inherit)
    straight!(sb2; length = 1.0)
    seal!(sb2)
    return build([sb1, sb2])
end

variant_row(
    [("dest = (1.0, 0, 1.5)", jumpto_after_bend_path((1.0, 0.0, 1.5)), [3]),
     ("dest = (1.3, 0, 1.5)", jumpto_after_bend_path((1.3, 0.0, 1.5)), [3]),
     ("dest = (0.5, 0, 2.0)", jumpto_after_bend_path((0.5, 0.0, 2.0)), [3])];
    spacing = 3.5, title = "JumpTo after 90° bend — point in the GLOBAL frame")

### 8.5 · G2 inheritance from the preceding segment

A bend flows directly into a `jumpby!` with no explicit curvature spec: the
connector inherits the bend's exit curvature κ = 1/R (G2 continuity by default).
Smaller R launches the connector on a visibly tighter initial arc.

In [ ]:
function g2_inheritance_path(R_bend)
    sb = SubpathBuilder(); start!(sb)
    bend!(sb; radius = R_bend, angle = π / 4, axis_angle = 0.0)
    jumpby!(sb; delta = (0.3, 0.0, 0.3))
    straight!(sb; length = 1.0)
    seal!(sb)
    return build(sb)
end

variant_row(
    [("R = 0.30 (κ_in ≈ 3.33)", g2_inheritance_path(0.30), [2]),
     ("R = 0.50 (κ_in = 2.00)", g2_inheritance_path(0.50), [2]),
     ("R = 0.80 (κ_in = 1.25)", g2_inheritance_path(0.80), [2])];
    spacing = 3.0, title = "JumpBy after a bend — G2 inheritance of incoming κ")

### 8.6 · Routing in 3D

In [ ]:
variant_row([("composite", routing, [2, 4, 6])];
            spacing = 0.0, title = "JumpTo routing — serpentine (3D view)")

## 9 · MCM temperature PTF — end-to-end uncertainty

The flagship physics demo. Bend birefringence scales as 1/R², so at R = 2.5 cm a
~10000-turn helix accumulates |Δβ|·L ≈ 1775·2π rad of retardation. The fractional
turn count (10001.892…) is tuned so that at T = 30 °C the first helix sits exactly
at **mid-fringe** (mod(Γ, 2π) = π) — the operating point of maximum temperature
sensitivity (at a crossing, dPTF/dT = 0). Over ±5 °C the retardation swings ≈ ±0.75π.

Temperature enters **twice**, deliberately: as `MCMadd(:T_K, ΔT)` meta on the first
helix (geometric thermal expansion via the cladding CTE, baked in by `Fiber`) and as
`T_ref_K = T_K` on the final `Fiber` binding (temperature-dependent material
indices). One `propagate_fiber` call propagates the whole ensemble; per-particle
Jones matrices are then sliced out and converted to Stokes observables
(`stokes_ensemble` in the helper) for input state H.

The output stays nearly linearly polarized (DLP ≈ 1, S3 ≈ 0): the helix
birefringence rotates the linear polarization **angle** without adding ellipticity —
the angle is the sensitive observable, not DLP.

In [ ]:
using Distributions

MCM_XS = StepIndexCrossSection(
    SilicaGermaniaGlass(0.036),
    SilicaGermaniaGlass(0.0),
    8.2e-6,
    125e-6,
)
MCM_T_REF = 303.15      # 30 °C design point
MCM_λ     = 1550e-9

# turns chosen so Δβ(30 °C)·L is an odd multiple of π (mid-fringe).
HELIX1_TURNS = 10001.892069208387

# 5-segment fiber with sinusoidal material spin over the whole Subpath.
# ΔT_K is the temperature offset applied to the first helix via :T_K meta.
function mcm_fiber(ΔT_K)
    sb = SubpathBuilder(); start!(sb; spin_rate = s -> sin(2π * s / 100.0))
    straight!(sb; length = 5.0, meta = [Nickname("lead-in")])
    helix!(sb; radius = 0.025, pitch = 0.05, turns = HELIX1_TURNS, axis_angle = 0.0,
           meta = AbstractMeta[Nickname("temperature-sensitive helix"),
                               MCMadd(:T_K, ΔT_K)])
    straight!(sb; length = 5.0, meta = [Nickname("spacer")])
    helix!(sb; radius = 0.025, pitch = 0.05, turns = 10000.0, axis_angle = 0.0,
           meta = [Nickname("reference helix")])
    straight!(sb; length = 5.0, meta = [Nickname("lead-out")])
    seal!(sb)
    return Fiber(sb; cross_section = MCM_XS, T_ref_K = MCM_T_REF)
end
println("mcm_fiber defined; total ≈ ", round(path_length(mcm_fiber(0.0).path); digits = 1),
        " m of fiber")

### 9.1 · StaticParticles(50): angle and Stokes vs temperature

T ~ N(30 °C, 5 °C) with 50 SIMD-friendly particles — one ensemble propagation.

In [ ]:
MonteCarloMeasurements.unsafe_comparisons(true)
T_C  = StaticParticles(50, Normal(30.0, 5.0))
T_K  = T_C + 273.15
fiber_thermal = mcm_fiber(T_K - MCM_T_REF)       # geometric thermal expansion
fiber = Fiber(fiber_thermal.path; cross_section = MCM_XS, T_ref_K = T_K)
J, _ = propagate_fiber(fiber; λ_m = MCM_λ)
MonteCarloMeasurements.unsafe_comparisons(false)

st = stokes_ensemble(J)
display(ensemble_scatter(T_C.particles,
    [("pol. angle", st.angle_deg)];
    ylab = "angle (deg)", yrange = [0, 180],
    title = "Linear polarization angle vs temperature — StaticParticles(50)"))
ensemble_scatter(T_C.particles,
    [("S1", st.s1), ("S2", st.s2), ("S3", st.s3), ("DLP", st.dlp)];
    ylab = "Stokes / DLP", yrange = [-1.05, 1.05],
    title = "Stokes parameters vs temperature — DLP ≈ 1, S3 ≈ 0")

### 9.2 · Particles(500): ensemble scatter on the Poincaré equator

One heap-backed `Particles` run; the ensemble traces an arc on the S1–S2 equatorial
projection, parameterized by temperature. *(The legacy demo used 2000 particles;
500 preserves the story at a quarter of the runtime.)*

In [ ]:
MonteCarloMeasurements.unsafe_comparisons(true)
T_C  = Particles(500, Normal(30.0, 5.0))
T_K  = T_C + 273.15
fiber_thermal = mcm_fiber(T_K - MCM_T_REF)
fiber = Fiber(fiber_thermal.path; cross_section = MCM_XS, T_ref_K = T_K)
J, _ = propagate_fiber(fiber; λ_m = MCM_λ)
MonteCarloMeasurements.unsafe_comparisons(false)

st = stokes_ensemble(J)
display(ensemble_scatter(T_C.particles,
    [("pol. angle", st.angle_deg)];
    ylab = "angle (deg)", yrange = [0, 180],
    title = "Linear polarization angle vs temperature — Particles(500)"))
poincare_equatorial(st.s1, st.s2; color = T_C.particles,
    title = "Poincaré equatorial projection (S1–S2) — ensemble arc vs temperature")

## 10 · MCM speed benchmarks

How much does ensemble propagation cost relative to a nominal `Float64` run, and
where is the `StaticParticles` SIMD sweet spot? Two timings per case: **first-call**
(JIT compilation + one run) and **steady-state** (BenchmarkTools minimum over a few
post-JIT samples). Numbers are machine-dependent — nothing here is asserted.

*Scaling note: the benchmark fiber keeps the §9 five-segment structure but with
100-turn helices (~100× shorter than the legacy ~10000-turn, mid-fringe-tuned fiber;
that tuning matters for sensitivity, not speed) so the repeated timed runs — in
particular the Particles(2000) case — stay tractable. Raise `turns` in `bench_fiber`
on faster hardware.*

In [ ]:
using BenchmarkTools

function bench_fiber(ΔT_K)
    sb = SubpathBuilder(); start!(sb; spin_rate = s -> sin(2π * s / 100.0))
    straight!(sb; length = 5.0)
    helix!(sb; radius = 0.025, pitch = 0.05, turns = 100.0, axis_angle = 0.0,
           meta = AbstractMeta[MCMadd(:T_K, ΔT_K)])
    straight!(sb; length = 5.0)
    helix!(sb; radius = 0.025, pitch = 0.05, turns = 100.0, axis_angle = 0.0)
    straight!(sb; length = 5.0)
    seal!(sb)
    return Fiber(sb; cross_section = MCM_XS, T_ref_K = MCM_T_REF)
end

first_call_ms(f) = (t0 = time_ns(); f(); (time_ns() - t0) / 1e6)

bench_cases = [
    ("Float64",             0,    () -> 30.0),
    ("Particles(2000)",     2000, () -> Particles(2000, Normal(30.0, 5.0))),
    ("StaticParticles(50)", 50,   () -> StaticParticles(50, Normal(30.0, 5.0))),
    ("StaticParticles(75)", 75,   () -> StaticParticles(75, Normal(30.0, 5.0))),
]
println("benchmark scaffolding defined")

### 10.1 · `propagate_fiber` alone (fiber pre-built)

In [ ]:
MonteCarloMeasurements.unsafe_comparisons(true)
results = map(bench_cases) do (label, n_p, make_T)
    T_K = make_T() + 273.15
    fiber = Fiber(bench_fiber(T_K - MCM_T_REF).path;
                  cross_section = MCM_XS, T_ref_K = T_K)
    fn() = propagate_fiber(fiber; λ_m = MCM_λ)
    first  = first_call_ms(fn)
    steady = @belapsed($fn(); samples = 3, evals = 1) * 1e3
    println(rpad(label, 22), "first ", lpad(round(first; digits = 1), 10), " ms",
            "   steady ", lpad(round(steady; digits = 1), 10), " ms")
    (; label, n_particles = n_p, first_ms = first, steady_ms = steady)
end
MonteCarloMeasurements.unsafe_comparisons(false)

display(benchmark_chart(collect(results);
    title = "propagate_fiber — first-call vs steady-state (log scale)"))
benchmark_table(collect(results))

### 10.2 · Build + propagate (the full per-sample pipeline)

Timing includes the `:T_K` thermal geometry build, `Fiber` construction, and
propagation.

In [ ]:
MonteCarloMeasurements.unsafe_comparisons(true)
results = map(bench_cases) do (label, n_p, make_T)
    T_K = make_T() + 273.15
    fn() = begin
        f2 = Fiber(bench_fiber(T_K - MCM_T_REF).path;
                   cross_section = MCM_XS, T_ref_K = T_K)
        propagate_fiber(f2; λ_m = MCM_λ)
    end
    first  = first_call_ms(fn)
    steady = @belapsed($fn(); samples = 3, evals = 1) * 1e3
    println(rpad(label, 22), "first ", lpad(round(first; digits = 1), 10), " ms",
            "   steady ", lpad(round(steady; digits = 1), 10), " ms")
    (; label, n_particles = n_p, first_ms = first, steady_ms = steady)
end
MonteCarloMeasurements.unsafe_comparisons(false)

display(benchmark_chart(collect(results);
    title = "build + propagate_fiber — first-call vs steady-state (log scale)"))
benchmark_table(collect(results))